In [1]:
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0
    
    def __lt__(self, other):
        return self.f < other.f

def heuristic(current_pos, end_pos):
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])

def best_first_search_multiple_goals(maze, start, goals):
    rows, cols = len(maze), len(maze[0])
    all_goals = goals.copy()
    current_position = start
    full_path = [start]
    total_cost = 0
    
    while all_goals:
        nearest_goal = None
        min_distance = float('inf')
        
        for goal in all_goals:
            dist = heuristic(current_position, goal)
            if dist < min_distance:
                min_distance = dist
                nearest_goal = goal
        
        path_to_goal = best_first_search_single(maze, current_position, nearest_goal)
        
        if path_to_goal:
            if len(path_to_goal) > 1:
                full_path.extend(path_to_goal[1:])
                total_cost += len(path_to_goal) - 1
            
            current_position = nearest_goal
            all_goals.remove(nearest_goal)
        else:
            return None, float('inf')
    
    return full_path, total_cost

def best_first_search_single(maze, start, end):
    rows, cols = len(maze), len(maze[0])
    start_node = Node(start)
    end_node = Node(end)
    
    frontier = PriorityQueue()
    frontier.put(start_node)
    visited = set()
    
    while not frontier.empty():
        current_node = frontier.get()
        current_pos = current_node.position
        
        if current_pos == end:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1]
        
        visited.add(current_pos)
        
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)
            
            if (0 <= new_pos[0] < rows and 
                0 <= new_pos[1] < cols and 
                maze[new_pos[0]][new_pos[1]] == 0 and 
                new_pos not in visited):
                
                new_node = Node(new_pos, current_node)
                new_node.g = current_node.g + 1
                new_node.h = heuristic(new_pos, end)
                new_node.f = new_node.h
                
                frontier.put(new_node)
                visited.add(new_pos)
    
    return None

maze = [
    [0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0],
    [1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 1, 0, 0, 0, 1, 0]
]

start = (0, 0)
goals = [(2, 7), (5, 7), (7, 3), (4, 4)]

print("TASK #1: Enhanced Maze Navigation with Multiple Goals")
print(f"Start: {start}")
print(f"Goals: {goals}")

path, cost = best_first_search_multiple_goals(maze, start, goals)

if path:
    print(f"Path visiting all goals: {path}")
    print(f"Total steps: {cost}")
else:
    print("No path found to all goals!")
print()

TASK #1: Enhanced Maze Navigation with Multiple Goals
Start: (0, 0)
Goals: [(2, 7), (5, 7), (7, 3), (4, 4)]
Path visiting all goals: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4), (4, 5), (5, 5), (5, 6), (5, 7), (4, 7), (3, 7), (2, 7), (3, 7), (4, 7), (5, 7), (5, 6), (5, 5), (6, 5), (7, 5), (7, 4), (7, 3)]
Total steps: 24



In [5]:
import random
import time
from queue import PriorityQueue

class DynamicAStar:
    def __init__(self, graph, heuristic, base_costs):
        self.graph = graph
        self.heuristic = heuristic
        self.base_costs = base_costs
        self.current_costs = self.copy_costs(base_costs)
        self.last_update_time = time.time()
        self.update_interval = 3
        
    def copy_costs(self, costs):
        new_costs = {}
        for node, neighbors in costs.items():
            new_costs[node] = neighbors.copy()
        return new_costs
    
    def update_costs_randomly(self):
        current_time = time.time()
        if current_time - self.last_update_time >= self.update_interval:
            for node in self.current_costs:
                for neighbor in self.current_costs[node]:
                    change_factor = random.uniform(0.7, 1.5)
                    original_cost = self.base_costs[node][neighbor]
                    new_cost = round(original_cost * change_factor, 1)
                    self.current_costs[node][neighbor] = new_cost
            
            self.last_update_time = current_time
            return True
        return False
    
    def a_star_search(self, start, goal, max_iterations=100):
        frontier = PriorityQueue()
        frontier.put((self.heuristic[start], start, [start]))
        g_costs = {start: 0}
        visited = set()
        iterations = 0
        
        while not frontier.empty() and iterations < max_iterations:
            iterations += 1
            self.update_costs_randomly()
            
            f_cost, current_node, path = frontier.get()
            
            if current_node in visited:
                continue
            
            g_cost = g_costs[current_node]
            
            if current_node == goal:
                return path, g_cost
            
            visited.add(current_node)
            
            for neighbor, edge_cost in self.current_costs[current_node].items():
                if neighbor not in visited:
                    new_g_cost = g_cost + edge_cost
                    
                    if neighbor not in g_costs or new_g_cost < g_costs[neighbor]:
                        g_costs[neighbor] = new_g_cost
                        f_cost = new_g_cost + self.heuristic[neighbor]
                        frontier.put((f_cost, neighbor, path + [neighbor]))
            
            time.sleep(1)
        
        return None, float('inf')
    
    def adaptive_a_star(self, start, goal):
        initial_path, initial_cost = self.a_star_search(start, goal)
        
        if initial_path:
            best_path = initial_path
            best_cost = initial_cost
            
            monitor_time = 10
            start_time = time.time()
            
            while time.time() - start_time < monitor_time:
                if self.update_costs_randomly():
                    new_path, new_cost = self.a_star_search(start, goal, max_iterations=50)
                    
                    if new_path and new_cost < best_cost:
                        best_path = new_path
                        best_cost = new_cost
                
                time.sleep(1)
            
            return best_path, best_cost
        else:
            return None, float('inf')

graph = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

base_costs = {
    'A': {'B': 4, 'C': 3},
    'B': {'E': 12, 'F': 5},
    'C': {'D': 7, 'E': 10},
    'D': {'E': 2},
    'E': {'G': 5},
    'F': {'G': 16},
    'G': {}
}

heuristic = {'A': 14, 'B': 12, 'C': 11, 'D': 6, 'E': 4, 'F': 11, 'G': 0}

dynamic_astar = DynamicAStar(graph, heuristic, base_costs)

path, cost = dynamic_astar.adaptive_a_star('A', 'G')

if path:
    print(f"Optimal path: {' -> '.join(path)}")
    print(f"Total cost: {cost}")
print()

TASK #2: A* Search with Dynamically Changing Edge Costs
Optimal path: A -> C -> D -> E -> G
Total cost: 16.9



In [7]:
from queue import PriorityQueue

class DeliveryPoint:
    def __init__(self, id, location, time_window_start, time_window_end, priority=1):
        self.id = id
        self.location = location
        self.time_window_start = time_window_start
        self.time_window_end = time_window_end
        self.priority = priority
        self.visited = False
    
    def __str__(self):
        return f"Delivery {self.id} at {self.location} [{self.time_window_start}-{self.time_window_end}]"

def manhattan_distance(pos1, pos2):
    return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

class DeliveryOptimizer:
    def __init__(self, depot_location):
        self.depot = depot_location
        self.deliveries = []
        self.current_time = 0
        self.total_distance = 0
        self.route = [depot_location]
        
    def add_delivery(self, delivery_point):
        self.deliveries.append(delivery_point)
    
    def calculate_urgency_score(self, delivery, current_location, current_time):
        distance = manhattan_distance(current_location, delivery.location)
        travel_time = distance
        time_remaining = delivery.time_window_end - current_time
        
        if time_remaining <= 0:
            urgency = 1000 + (100 - delivery.priority * 10)
        elif time_remaining < travel_time:
            urgency = 500 + (travel_time - time_remaining) * 10
        else:
            window_flexibility = time_remaining - travel_time
            urgency = (delivery.priority * 50) - (window_flexibility * 2)
        
        return urgency
    
    def greedy_bfs_with_time_windows(self):
        current_location = self.depot
        current_time = 0
        route = [self.depot]
        unvisited = self.deliveries.copy()
        total_distance = 0
        visited_deliveries = []
        
        while unvisited:
            candidates = []
            for delivery in unvisited:
                urgency = self.calculate_urgency_score(delivery, current_location, current_time)
                distance = manhattan_distance(current_location, delivery.location)
                candidates.append((urgency, distance, delivery))
            
            candidates.sort(reverse=True)
            urgency, distance, next_delivery = candidates[0]
            
            arrival_time = current_time + distance
            
            if arrival_time < next_delivery.time_window_start:
                arrival_time = next_delivery.time_window_start
            
            route.append(next_delivery.location)
            current_location = next_delivery.location
            current_time = arrival_time + 1
            total_distance += distance
            visited_deliveries.append(next_delivery)
            unvisited.remove(next_delivery)
        
        return_distance = manhattan_distance(current_location, self.depot)
        total_distance += return_distance
        route.append(self.depot)
        
        return route, total_distance, visited_deliveries

optimizer = DeliveryOptimizer((0, 0))

optimizer.add_delivery(DeliveryPoint('A', (2, 3), 10, 20, priority=3))
optimizer.add_delivery(DeliveryPoint('B', (5, 1), 5, 15, priority=2))
optimizer.add_delivery(DeliveryPoint('C', (3, 5), 25, 35, priority=1))
optimizer.add_delivery(DeliveryPoint('D', (7, 2), 15, 25, priority=3))
optimizer.add_delivery(DeliveryPoint('E', (4, 7), 30, 45, priority=2))
optimizer.add_delivery(DeliveryPoint('F', (6, 4), 20, 30, priority=1))

print(f"Depot: {optimizer.depot}")
print("Delivery Points:")
for d in optimizer.deliveries:
    print(f"  {d}")

route, distance, visited = optimizer.greedy_bfs_with_time_windows()

print(f"\nOptimal route: {route}")
print(f"Total distance: {distance} units")
print(f"Delivery order: {[d.id for d in visited]}")
print()

Depot: (0, 0)
Delivery Points:
  Delivery A at (2, 3) [10-20]
  Delivery B at (5, 1) [5-15]
  Delivery C at (3, 5) [25-35]
  Delivery D at (7, 2) [15-25]
  Delivery E at (4, 7) [30-45]
  Delivery F at (6, 4) [20-30]

Optimal route: [(0, 0), (2, 3), (5, 1), (7, 2), (4, 7), (6, 4), (3, 5), (0, 0)]
Total distance: 38 units
Delivery order: ['A', 'B', 'D', 'E', 'F', 'C']

